In [ ]:
%pip install --upgrade mlflow databricks-sdk dspy openai
dbutils.library.restartPython()

# Cambio de Modelo: No es tan simple como intercambiar prompts

A continuación se presenta un ejemplo que actualiza un prompt para un modelo diferente. Es un prompt simple **clasificar el riesgo fiscal** por lo que probablemente verá mejoras mayores para casos de uso más complejos. Pasaremos de GPT-5 a Gemma 3/GPT-OSS-20B

Este notebook está diseñado para clasificar descripciones de operaciones fiscales en categorías de riesgo para el Servicio de Administración Tributaria (SAT) de México.

Asegúrese de tener acceso a las APIs de Databricks Foundation Model para ejecutar esto exitosamente.

In [ ]:
import mlflow
import openai
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.scorers import Correctness
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Cambie el catálogo y esquema a su catálogo y esquema
catalog = "main"
schema = "default"
prompt_registry_name = "clasificador_riesgo_fiscal_sat"
prompt_location = f"{catalog}.{schema}.{prompt_registry_name}"

openai_client = w.serving_endpoints.get_open_ai_client()

# Registrar el prompt inicial
prompt = mlflow.genai.register_prompt(
    name=prompt_location,
    template="Clasifica el nivel de riesgo fiscal de la siguiente operación: {{descripcion}}",
)


# Definir la función de predicción
def predict_fn(descripcion: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}/1")
    completion = openai_client.chat.completions.create(
        model="databricks-gpt-5",
        # Cargar plantilla de prompt usando PromptVersion.format()
        messages=[{"role": "user", "content": prompt.format(descripcion=descripcion)}],
    )
    return completion.choices[0].message.content

# Pruebe su Función

Observe qué tan precisamente el modelo puede clasificar la entrada con un prompt básico. Aunque sea preciso, no está alineado a ninguna tarea o caso de uso específico que estamos buscando para el SAT.

In [ ]:
from IPython.display import Markdown

# Ejemplo de operación fiscal sospechosa
output = predict_fn("Empresa con ingresos declarados de 50 millones de pesos pero deducciones por conceptos de servicios profesionales que representan el 95% de sus ingresos, pagados a empresas relacionadas en el extranjero sin sustancia económica aparente.")

Markdown(output)

# Optimización contra Datos

Ahora proporcionaremos algunos datos con respuestas esperadas y hechos. Esto ayudará a optimizar nuestro modelo para comportarse y producir resultados que se ajusten a nuestros casos de uso del SAT.

En este caso, queremos que el modelo produzca una palabra de una selección de cinco categorías de riesgo. Solo debe producir esa palabra sin ninguna explicación adicional.

**Categorías de Riesgo:**
- ALTO_RIESGO: Operaciones con indicios claros de evasión o fraude fiscal
- MEDIO_RIESGO: Operaciones que requieren revisión detallada
- BAJO_RIESGO: Operaciones con anomalías menores
- SIN_RIESGO: Operaciones fiscales normales
- REQUIERE_INVESTIGACION: Casos que necesitan auditoría especializada

In [ ]:
# Datos de entrenamiento con entradas y salidas esperadas
dataset = [
    {
        "inputs": {"descripcion": "Empresa con ingresos declarados de 50 millones de pesos pero deducciones por conceptos de servicios profesionales que representan el 95% de sus ingresos, pagados a empresas relacionadas en el extranjero sin sustancia económica aparente."},
        "expectations": {"expected_facts": ["La clasificación debe ser 'ALTO_RIESGO', 'MEDIO_RIESGO', 'BAJO_RIESGO', 'SIN_RIESGO', 'REQUIERE_INVESTIGACION'"]}
    },
    {
        "inputs": {"descripcion": "Empresa con ingresos declarados de 50 millones de pesos pero deducciones por conceptos de servicios profesionales que representan el 95% de sus ingresos, pagados a empresas relacionadas en el extranjero sin sustancia económica aparente."},
        "outputs": {"response": "ALTO_RIESGO"},
        "expectations": {"expected_response": "ALTO_RIESGO"}
    },
    {
        "inputs": {"descripcion": "Contribuyente persona física con actividad empresarial que declara ingresos consistentes con su giro comercial, deducciones proporcionales y pagos de impuestos en tiempo y forma durante los últimos 5 años."},
        "expectations": {"expected_facts": ["La clasificación debe ser 'ALTO_RIESGO', 'MEDIO_RIESGO', 'BAJO_RIESGO', 'SIN_RIESGO', 'REQUIERE_INVESTIGACION'"]}
    },
    {
        "inputs": {"descripcion": "Contribuyente persona física con actividad empresarial que declara ingresos consistentes con su giro comercial, deducciones proporcionales y pagos de impuestos en tiempo y forma durante los últimos 5 años."},
        "outputs": {"response": "SIN_RIESGO"},
        "expectations": {"expected_response": "SIN_RIESGO"}
    },
    {
        "inputs": {"descripcion": "Empresa que reporta operaciones con facturas de proveedores que aparecen en la lista del artículo 69-B del Código Fiscal de la Federación como Empresas que Facturan Operaciones Simuladas (EFOS)."},
        "expectations": {"expected_facts": ["La clasificación debe ser 'ALTO_RIESGO', 'MEDIO_RIESGO', 'BAJO_RIESGO', 'SIN_RIESGO', 'REQUIERE_INVESTIGACION'"]}
    },
    {
        "inputs": {"descripcion": "Empresa que reporta operaciones con facturas de proveedores que aparecen en la lista del artículo 69-B del Código Fiscal de la Federación como Empresas que Facturan Operaciones Simuladas (EFOS)."},
        "outputs": {"response": "ALTO_RIESGO"},
        "expectations": {"expected_response": "ALTO_RIESGO"}
    },
    {
        "inputs": {"descripcion": "Pequeño comercio que presenta una variación del 15% en sus ingresos respecto al año anterior debido a remodelación del local, con documentación soporte completa."},
        "expectations": {"expected_facts": ["La clasificación debe ser 'ALTO_RIESGO', 'MEDIO_RIESGO', 'BAJO_RIESGO', 'SIN_RIESGO', 'REQUIERE_INVESTIGACION'"]}
    },
    {
        "inputs": {"descripcion": "Pequeño comercio que presenta una variación del 15% en sus ingresos respecto al año anterior debido a remodelación del local, con documentación soporte completa."},
        "outputs": {"response": "BAJO_RIESGO"},
        "expectations": {"expected_response": "BAJO_RIESGO"}
    },
    {
        "inputs": {"descripcion": "Empresa constructora con múltiples contratos gubernamentales que presenta subcontrataciones en cascada con empresas recién constituidas y sin historial crediticio."},
        "expectations": {"expected_facts": ["La clasificación debe ser 'ALTO_RIESGO', 'MEDIO_RIESGO', 'BAJO_RIESGO', 'SIN_RIESGO', 'REQUIERE_INVESTIGACION'"]}
    },
    {
        "inputs": {"descripcion": "Empresa constructora con múltiples contratos gubernamentales que presenta subcontrataciones en cascada con empresas recién constituidas y sin historial crediticio."},
        "outputs": {"response": "REQUIERE_INVESTIGACION"},
        "expectations": {"expected_response": "REQUIERE_INVESTIGACION"}
    },
    {
        "inputs": {"descripcion": "Importador que declara mercancías con valor inferior al precio de mercado en un 40%, con origen en países considerados de bajo costo pero sin justificación comercial."},
        "expectations": {"expected_facts": ["La clasificación debe ser 'ALTO_RIESGO', 'MEDIO_RIESGO', 'BAJO_RIESGO', 'SIN_RIESGO', 'REQUIERE_INVESTIGACION'"]}
    },
    {
        "inputs": {"descripcion": "Importador que declara mercancías con valor inferior al precio de mercado en un 40%, con origen en países considerados de bajo costo pero sin justificación comercial."},
        "outputs": {"response": "MEDIO_RIESGO"},
        "expectations": {"expected_response": "MEDIO_RIESGO"}
    }
]

# Optimizar el prompt
result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=dataset,
    prompt_uris=[prompt.uri],
    optimizer=GepaPromptOptimizer(reflection_model="databricks:/databricks-gpt-5-2"),
    scorers=[Correctness(model="databricks:/databricks-gpt-5")],
)

# Usar el prompt optimizado
optimized_prompt = result.optimized_prompts[0]
print(f"Plantilla optimizada: {optimized_prompt.template}")

In [ ]:
print(f"Puntuación Inicial: {result.initial_eval_score}\n") 
print(f"Puntuación Final: {result.final_eval_score}")

# Revisemos los Cambios

Probemos cómo funciona ahora GPT-OSS con el prompt optimizado para clasificación de riesgo fiscal.

In [ ]:
import mlflow

mlflow.openai.autolog()

def predict_fn(descripcion: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}/2") 
    completion = openai_client.chat.completions.create(
        model = "databricks-gpt-5",
        messages=[{"role": "system", "content": prompt.format(descripcion=descripcion)}]
    )
    return completion.choices[0].message.content

In [ ]:
# Caso de prueba: Operación sospechosa de lavado de dinero
output = predict_fn(descripcion="Persona moral del sector inmobiliario que recibe múltiples depósitos en efectivo fraccionados justo por debajo del umbral de reporte, provenientes de personas físicas sin relación comercial aparente.")

In [ ]:
# Respuesta correcta: ALTO_RIESGO
output

# Probemos con Gemma

¿Qué tan bien funciona con este prompt optimizado para un modelo más pequeño?

In [ ]:
from IPython.display import Markdown
prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}/2")

Markdown(prompt.template)

In [ ]:
def predict_fn_gemma(descripcion: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}/2")
    completion = openai_client.chat.completions.create(
        model="databricks-gemma-3-12b",
        messages=[{"role": "system", "content": prompt.format(descripcion=descripcion)},
            {"role": "user", "content": descripcion}],
    )
    return completion.choices[0].message.content

In [ ]:
# Caso de prueba con Gemma
output = predict_fn_gemma(descripcion="Persona moral del sector inmobiliario que recibe múltiples depósitos en efectivo fraccionados justo por debajo del umbral de reporte, provenientes de personas físicas sin relación comercial aparente.")

In [ ]:
# Respuesta correcta: ALTO_RIESGO
output

# Resultado Incorrecto, No Sorprende

Aunque cambiamos a un modelo más pequeño, podemos ver que no es tan simple como darle un prompt optimizado y esperar las mismas mejoras de rendimiento.

Debemos re-optimizar para obtener un nuevo prompt para el nuevo modelo basándonos en el prompt original.

Hagámoslo a continuación. Configuraré una nueva función para usar el modelo Gemma 3.

In [ ]:
def predict_fn_gemma(descripcion: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}/1")
    completion = openai_client.chat.completions.create(
        model="databricks-gemma-3-12b",
        messages=[{"role": "user", "content": prompt.format(descripcion=descripcion)}],
    )
    return completion.choices[0].message.content

In [ ]:
result_gemma_oss = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn_gemma,
    train_data=dataset,
    prompt_uris=[prompt.uri],
    optimizer=GepaPromptOptimizer(reflection_model="databricks:/databricks-gpt-5-2"),
    scorers=[Correctness(model="databricks:/databricks-gpt-5")],
)
# Usar el prompt optimizado
optimized_prompt = result.optimized_prompts[0]
print(f"Plantilla optimizada: {optimized_prompt.template}")

# Ya deberíamos tener puntuaciones decentes

In [ ]:
print(f"Puntuación Inicial: {result_gemma_oss.initial_eval_score}\n") 
print(f"Puntuación Final: {result_gemma_oss.final_eval_score}")

In [ ]:
from IPython.display import Markdown
prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}/3")

Markdown(prompt.template)

# Verifiquemos de nuevo

In [ ]:
def predict_fn_gemma_updated(descripcion: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}/3")
    completion = openai_client.chat.completions.create(
        model="databricks-gemma-3-12b",
        messages=[{"role": "system", "content": prompt.format(descripcion=descripcion)},
            {"role": "user", "content": descripcion}],
    )
    return completion.choices[0].message.content

In [ ]:
# Nuevo caso de prueba: Transferencias internacionales sospechosas
output = predict_fn_gemma_updated(descripcion="Empresa mexicana que realiza pagos recurrentes a empresas en jurisdicciones de baja tributación por concepto de regalías y asistencia técnica sin evidencia de servicios recibidos.")

In [ ]:
# Respuesta correcta: ALTO_RIESGO
output

# ¿Es ahora correcto?

No podemos asumir que el prompt existente funcionará para todos los modelos o será eficiente para todos los modelos. ¡Solo toma unos minutos re-optimizar el modelo!

Regresemos a nuestro experimento para agregar algunos alias.

Ahora podemos usar el registro de prompts de MLflow para cargar los prompts correctos dependiendo del modelo que queramos usar para clasificación de riesgo fiscal del SAT.

In [ ]:
# GPT-OSS 20B - Clasificador de Riesgo Fiscal

def predict_fn(descripcion: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}@gpt_oss_20b")
    completion = openai_client.chat.completions.create(
        model = "databricks-gpt-oss-20b",
        messages=[{"role": "system", "content": prompt.format(descripcion=descripcion)}]
    )
    return completion.choices[0].message.content

# Caso de prueba: Subfacturación de exportaciones
output = predict_fn(descripcion="Exportador que declara ventas al extranjero con precios significativamente menores a los precios de mercado, con clientes en países sin tratado de intercambio de información tributaria.")
output[1]['text']

In [ ]:
# GPT-5 - Clasificador de Riesgo Fiscal

def predict_fn(descripcion: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}@gpt_oss_20b")
    completion = openai_client.chat.completions.create(
        model = "databricks-gpt-5",
        messages=[{"role": "system", "content": prompt.format(descripcion=descripcion)}]
    )
    return completion.choices[0].message.content

# Caso de prueba: Operación legítima
output = predict_fn(descripcion="Empresa manufacturera con 20 años de operación que presenta declaraciones consistentes, auditorías limpias y cumplimiento de todas sus obligaciones fiscales federales y estatales.")
output

In [ ]:
# Gemma - Clasificador de Riesgo Fiscal para SAT

def predict_fn_gemma(descripcion: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_location}@gpt5")
    completion = openai_client.chat.completions.create(
        model="databricks-gemma-3-12b",
        messages=[{"role": "system", "content": prompt.format(descripcion=descripcion)},
            {"role": "user", "content": descripcion}],
    )
    return completion.choices[0].message.content

# Caso de prueba: Posible triangulación
output = predict_fn_gemma(descripcion="Comercializadora que compra productos a proveedores nacionales y los vende a empresas relacionadas en el extranjero a precios de costo, generando pérdidas fiscales recurrentes.")
output